In [ ]:
import os
import pandas as pd
from os.path import join

In [ ]:
src = os.path.join("data", "newsguard")
pd.read_csv(join(src, 'newsguard_twice_month.csv'), nrows=0).columns

In [3]:
df = pd.read_csv(join(src, 'newsguard_twice_month.csv'),
                 usecols=['Domain', 'Country', 
                          'file_date', 'Score',
                          'Orientation'])

In [4]:
# df.head(3)

In [5]:
df.file_date = pd.to_datetime(df.file_date, format='mixed')
df.file_date

0         2022-09-02 01:00:00
1         2022-09-02 01:00:00
2         2022-09-02 01:00:00
3         2022-09-02 01:00:00
4         2022-09-02 01:00:00
                  ...        
1054211   2024-09-15 01:00:00
1054212   2024-09-15 01:00:00
1054213   2024-09-15 01:00:00
1054214   2024-09-15 01:00:00
1054215   2024-09-15 01:00:00
Name: file_date, Length: 1054216, dtype: datetime64[ns]

In [6]:
# turn into month 
df['file_month'] = df.file_date.dt.to_period('M')
df.file_month

0          2022-09
1          2022-09
2          2022-09
3          2022-09
4          2022-09
            ...   
1054211    2024-09
1054212    2024-09
1054213    2024-09
1054214    2024-09
1054215    2024-09
Name: file_month, Length: 1054216, dtype: period[M]

In [7]:
# order by Domain and file_date
df = df.sort_values(['Domain', 'file_month', 'Score', 'Orientation'])
# df.head(6)

In [8]:
df.Orientation.value_counts()

Orientation
Right             125899
Far Right          82120
Slightly Right     57155
Slightly Left      32918
Left               24285
Far Left           13502
Name: count, dtype: int64

In [9]:
# recode "Slightly Left" to "Left"
df.Orientation = df.Orientation\
    .replace({
        'Far Right': 'Right',
        'Slightly Right': 'Right',
        'Slightly Left': 'Left',
        'Far Left': 'Left'
    })
df.Orientation.value_counts()

Orientation
Right    265174
Left      70705
Name: count, dtype: int64

In [10]:
# replace NaN with 'Neutral'
df.Orientation = df.Orientation.fillna('Neutral')
df.Orientation.value_counts()

Orientation
Neutral    718337
Right      265174
Left        70705
Name: count, dtype: int64

In [11]:
# remove duplicates
df_unique = df.drop_duplicates(subset=['Domain', 'file_month'],
                               keep='first') # keep first entry
len(df_unique)

471131

In [12]:
df_unique.Orientation.value_counts()

Orientation
Neutral    312068
Right      128158
Left        30905
Name: count, dtype: int64

In [13]:
df_unique.to_csv(join(src, 'domains_unique.csv'), index=False)

In [14]:
df_austria = df_unique[df_unique['Country'] == 'AT'].drop_duplicates(subset='Domain')
# df_austria

In [15]:
# save as csv
df_austria.to_csv(join(src, 'domains_austria.csv'), index=False)